# Language-Model Adapter — DIMER Artifact Inference Tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/language-model-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/language-model-pipeline/blob/main/tutorials/language_model_artifact_inference_colab.ipynb)

**Profile:** `ARTIFACT-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification `1.0`

This notebook consumes an **externally supplied PEFT adapter ZIP** produced by the companion E2E workflow. It validates the archive before loading model state, inspects the adapter's model/revision/runtime provenance, resolves that identity against the canonical repository registry, reconstructs the serving state from the adapter plus the explicitly permitted pinned base-model acquisition, accepts new user input, generates with the repository inference API, and exports machine-readable predictions.

**No training or fine-tuning occurs in this notebook.** The adapter contains LoRA delta weights and tokenizer/template state; it is not a complete language model and requires its exact base model revision.

**By the end of this notebook you will be able to:**
- verify an externally supplied adapter ZIP before model loading;
- inspect its manifest, immutable base-model identity, producer runtime, training provenance, and artifact format;
- reconstruct base model + tokenizer + adapter from the artifact contract;
- validate and score a new prompt you provide;
- export predictions and inference provenance as JSONL/JSON.

**Trust boundary.** Archive path-safety and manifest hashes establish internal consistency, not sender authenticity: an attacker who replaces both files and manifest can make them agree. A whole-archive SHA-256 is useful only when obtained through a trusted independent channel. Path-safe extraction also does not make arbitrary executable serialization safe. This contract requires `adapter_model.safetensors`, JSON metadata, `trustRemoteCode=false`, and repository-pinned base acquisition; load adapters only from a trusted producer.

**This notebook does not demonstrate:** training, model selection, benchmark accuracy, calibrated confidence, safety validation, arbitrary checkpoints, pickle/PyTorch object deserialization, or production fitness.

Repository references: [README](../README.md) · [artifact contract](../ARTIFACT_SPEC.md) · [provenance contract](../PROVENANCE_SPEC.md) · [security](../SECURITY.md).

## Prerequisites and input contract

- **Artifact:** upload exactly one `dimer-language-model-adapter.zip` produced by the current E2E tutorial contract. It must contain `artifact-manifest.json`, `provenance.json`, `adapter_config.json`, `adapter_model.safetensors`, and tokenizer assets.
- **Runtime:** Google Colab or Jupyter with a CUDA GPU; CPU-only QLoRA-style base loading is not supported by this tutorial path.
- **Network:** the matching pinned base model is downloaded from its immutable Hugging Face revision because the adapter intentionally contains deltas rather than duplicate base weights. There is no automatic fallback to a different model/revision.
- **New input:** the inference stage uses an editable Colab `CUSTOM_PROMPT` field. The prompt stays in the notebook runtime and is sent only to the locally loaded model; it is not sent to an external inference service. Do not enter confidential, restricted, personal, or sensitive text in a hosted notebook unless you are authorized to place that data there.

The artifact upload is mandatory for this profile. Model loading does not begin until archive safety, manifest integrity, provenance, canonical identity, and runtime compatibility checks pass.

## 1. Install and verify the matching runtime

The consumer must use the same critical software versions recorded by the producer. User-space dependencies are pinned exactly. Torch is accelerator-coupled, so the notebook verifies the Colab-provided semantic version and records its exact CUDA build instead of replacing the GPU wheel.

In [ ]:
%pip -q install transformers==5.16.1 tokenizers==0.23.2 huggingface-hub==1.30.0 peft==0.20.0 accelerate==1.14.0 bitsandbytes==0.49.0 safetensors==0.8.0 datasets==4.8.5 pandas==2.3.3 PyYAML==6.0.3 Jinja2==3.1.6
%pip -q install --no-deps git+https://github.com/kurtvalcorza/language-model-pipeline.git@0fe85e3c7408a569ac4893f924f5bbc165deff0b

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch

from lmpipeline.tutorial_api import (
    assert_runtime_compatible,
    assert_tutorial_runtime,
    consume_adapter_archive,
    generate_reply,
    load_adapter_for_inference,
    resolve_artifact_model,
    sha256_file,
    validate_prompt,
)

RUNTIME = assert_tutorial_runtime()
if not torch.cuda.is_available():
    raise RuntimeError("Artifact inference requires a CUDA GPU for the supported 4-bit path. In Colab choose Runtime > Change runtime type > T4 GPU.")
print(json.dumps(RUNTIME, indent=2))

## 2. Upload and verify the external adapter artifact

The ZIP is supplied from **outside this notebook execution**. The repository API rejects absolute paths, `..` traversal, backslash-ambiguous paths, symlinks, duplicate members, and archives expanding beyond 512 MiB. It then requires exactly one manifest, verifies every listed file's byte count and SHA-256, rejects unexpected unlisted files, and requires safe-serialized adapter/provenance files before provenance is trusted.

If the producer gave you the ZIP's SHA-256 through a trusted independent channel, paste it into `EXPECTED_ARTIFACT_ZIP_SHA256`. A digest copied from the same untrusted delivery channel does not establish authenticity.

In [ ]:
EXPECTED_ARTIFACT_ZIP_SHA256 = "" # @param {type:"string"}

from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one adapter ZIP")
name, payload = next(iter(uploaded.items()))
if not name.lower().endswith(".zip"):
    raise ValueError("The external artifact must be a .zip archive")
ARCHIVE = Path("/content") / Path(name).name
ARCHIVE.write_bytes(payload)

ARTIFACT_ROOT, MANIFEST, PROVENANCE = consume_adapter_archive(
    ARCHIVE,
    extraction_root="/content/dimer-language-model-artifact",
    expected_archive_sha256=EXPECTED_ARTIFACT_ZIP_SHA256,
)
print({
    "archive": ARCHIVE.name,
    "archiveSha256": sha256_file(ARCHIVE),
    "format": MANIFEST["format"],
    "formatVersion": MANIFEST["formatVersion"],
    "manifestedFiles": len(MANIFEST["files"]),
})

## 3. Inspect provenance and establish runtime/model compatibility

The adapter must name a canonical `modelKey`, exact upstream `baseModel`, immutable 40-character `baseModelRevision`, artifact format/version, training configuration, dataset identity, and producer runtime. The repository resolves `modelKey` through its current canonical registry and requires the recorded model ID and revision to match that entry exactly.

Critical producer and consumer package versions (`torch`, `transformers`, `tokenizers`, `peft`, `bitsandbytes`, `safetensors`) must match for this tutorial contract. A mismatch fails rather than guessing that serialization/template behavior is compatible.

In [ ]:
ENTRY = resolve_artifact_model(PROVENANCE)
assert_runtime_compatible(PROVENANCE, RUNTIME)
print(json.dumps({
    "modelKey": ENTRY.key,
    "baseModel": ENTRY.model_id,
    "baseModelRevision": ENTRY.revision,
    "baseModelLicense": PROVENANCE.get("baseModelLicense"),
    "dataset": PROVENANCE.get("dataset"),
    "datasetDigest": PROVENANCE.get("datasetDigest"),
    "training": PROVENANCE.get("training"),
    "producerRuntime": PROVENANCE.get("runtime"),
}, indent=2))
print("canonical model identity and runtime compatibility: PASS")

## 4. Reconstruct the serving state from the artifact contract

The tokenizer/chat template comes from the verified adapter artifact, preserving the formatting used by the producer. The base model is acquired at **exactly** the canonical revision named by the adapter; `trust_remote_code=False` is enforced. `PeftModel.from_pretrained(..., is_trainable=False)` then attaches the verified safetensors adapter.

This network acquisition is explicit and permitted because the artifact format intentionally stores only PEFT deltas. It never falls back to `main`, another revision, or another model.

In [ ]:
MODEL, TOKENIZER = load_adapter_for_inference(
    ENTRY,
    artifact_root=ARTIFACT_ROOT,
)
print({
    "baseModel": ENTRY.model_id,
    "revision": ENTRY.revision,
    "adapterRoot": str(ARTIFACT_ROOT),
    "gpuMemoryGiB": round(torch.cuda.memory_allocated() / 1024**3, 2),
})

## 5. Validate new user input before generation

Edit `CUSTOM_PROMPT` to your own new instruction. This is the notebook's real new-input path; it is not one of the prompts used by the producing notebook's training/evaluation workflow. Before generation, the repository API renders the prompt with the artifact tokenizer/chat template and checks it against the artifact's effective `maxSequenceLength` ceiling.

Generation uses deterministic greedy decoding by default (`do_sample=False`). This is useful for a reproducible artifact check. For product use, decoding policy is application-owned and should be selected, evaluated, and recorded deliberately rather than treated as model confidence.

In [ ]:
CUSTOM_PROMPT = "Sumulat ng dalawang pangungusap tungkol sa responsableng paggamit ng AI." # @param {type:"string"}
MAX_NEW_TOKENS = 128 # @param {type:"integer"}

MAX_SEQUENCE_LENGTH = int((PROVENANCE.get("training") or {}).get("maxSequenceLength", 0))
if MAX_SEQUENCE_LENGTH <= 0 or MAX_SEQUENCE_LENGTH > int(ENTRY.max_sequence_length):
    raise ValueError("Artifact provenance has an invalid maxSequenceLength")
PROMPT_TOKENS = validate_prompt(
    TOKENIZER,
    CUSTOM_PROMPT,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
)
if MAX_NEW_TOKENS <= 0 or PROMPT_TOKENS + MAX_NEW_TOKENS > MAX_SEQUENCE_LENGTH:
    raise ValueError(
        f"prompt ({PROMPT_TOKENS}) + MAX_NEW_TOKENS ({MAX_NEW_TOKENS}) exceeds the {MAX_SEQUENCE_LENGTH}-token context ceiling"
    )
print({
    "promptTokens": PROMPT_TOKENS,
    "maxNewTokens": MAX_NEW_TOKENS,
    "contextCeiling": MAX_SEQUENCE_LENGTH,
})

## 6. Predict and export machine-readable results

The core generation call goes through the repository's public tutorial API. The result is displayed for the learner **and** written to `artifact_inference_predictions.jsonl`. A companion provenance JSON records the artifact digest, base model/revision, decoding settings, runtime, and notebook profile so the output can be interpreted without persisted notebook cells.

In [ ]:
DECODING = {"do_sample": False}
ANSWER = generate_reply(
    MODEL,
    TOKENIZER,
    CUSTOM_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    decoding=DECODING,
)
if not ANSWER:
    raise RuntimeError("The reconstructed adapter generated an empty response")

RESULT = {
    "inputId": "prompt-1",
    "prompt": CUSTOM_PROMPT,
    "promptTokens": PROMPT_TOKENS,
    "output": ANSWER,
    "modelId": ENTRY.model_id,
    "modelRevision": ENTRY.revision,
    "decoding": {"doSample": False, "maxNewTokens": MAX_NEW_TOKENS},
}
OUTPUT_JSONL = Path("/content/artifact_inference_predictions.jsonl")
OUTPUT_JSONL.write_text(json.dumps(RESULT, ensure_ascii=False) + "
")
OUTPUT_PROVENANCE = Path("/content/artifact_inference_provenance.json")
OUTPUT_PROVENANCE.write_text(json.dumps({
    "profile": "ARTIFACT-INFERENCE",
    "notebookSpecVersion": "1.0",
    "artifactSha256": sha256_file(ARCHIVE),
    "artifactFormat": MANIFEST["format"],
    "artifactFormatVersion": MANIFEST["formatVersion"],
    "baseModel": ENTRY.model_id,
    "baseModelRevision": ENTRY.revision,
    "runtime": RUNTIME,
    "decoding": RESULT["decoding"],
}, indent=2))
display(pd.DataFrame([RESULT])[["inputId", "prompt", "output"]])
print("wrote", OUTPUT_JSONL, "and", OUTPUT_PROVENANCE)

## 7. Optional stochastic decoding

Greedy decoding is the reproducible verification path. If you want to explore normal generative behavior, sampling parameters materially change outputs and must be explicit. Sampling scores are not calibrated probabilities and repeated generations are expected to differ. This optional cell does not replace the deterministic exported result above.

In [ ]:
RUN_SAMPLING_EXAMPLE = False # @param {type:"boolean"}
if RUN_SAMPLING_EXAMPLE:
    sampled = generate_reply(
        MODEL,
        TOKENIZER,
        CUSTOM_PROMPT,
        max_new_tokens=MAX_NEW_TOKENS,
        decoding={"do_sample": True, "temperature": 0.7, "top_p": 0.8, "top_k": 20},
    )
    print(sampled)

## Interpretation, limits, and troubleshooting

A successful top-to-bottom run establishes that the supplied external ZIP passed path/symlink/size/manifest checks, its model identity and immutable revision match the canonical registry, its critical producer runtime matches this consumer runtime, its tokenizer and PEFT adapter reconstruct against the required base model revision, a new user prompt passes context validation, generation succeeds through the repository API, and machine-readable predictions/provenance are written.

It does **not** establish sender authenticity unless you independently authenticated the artifact/digest, benchmark accuracy, factual correctness, safety, fairness, calibration, robustness, or production fitness. Manifest agreement alone is not a signature. A successful safetensors load proves the expected serialization boundary was consumable; it does not validate the behavior learned by the adapter.

Common failures are intentionally explicit: reject an archive on digest or manifest mismatch; reject unsafe paths/symlinks; reject a model/revision that differs from the canonical registry; rebuild the artifact under the current supported runtime when critical package versions differ; shorten the prompt or generation budget when the context ceiling is exceeded.

For release verification, execute this notebook from a clean runtime using an adapter produced externally by the companion E2E notebook at the candidate revision and record the tested commit SHA, environment, and outcome in the repository's tutorial release-verification record.